In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import numpy as np

In [ ]:
# Load datasets
gdf       = gpd.read_file("2123-Dataset/parking_density_Employee_Capita/parking_density_Employee_Capita.shp")
all_blocks = gpd.read_file("tl_2019_06_tabblock10/tl_2019_06_tabblock10.shp")

In [ ]:
# Static map: Parking Spaces per Employee
ax = gdf.plot(column='off_all_e', cmap='viridis', legend=True,
              scheme='quantiles', figsize=(12, 8))
ax.set_title("Off-Street Parking Spaces per Employee by Census Block Group")
ax.set_axis_off()

In [ ]:
# Interactive map: Total Parking Spaces and Parking Spaces per Employee
gdf.explore(
    column='off_all_e',
    cmap='viridis',
    scheme='quantiles',
    k=5,
    tooltip=['blkgrpid', 'off_all', 'off_all_e', 'population'],
    popup=True,
    tiles="CartoDB positron",
    style_kwds=dict(color="black", weight=0.5)
)

In [ ]:
# Project to UTM (meters) for all areal interpolation
gdf = gdf.to_crs(epsg=32610)
all_blocks = all_blocks.to_crs(epsg=32610)

In [ ]:
def areal_interpolation(bg_id, county_fips, min_area_ratio=0.001):
    """
    Distributes off-street parking from a block group down to individual census blocks
    using areal interpolation. Displays a table and interactive map.

    Parameters:
    - bg_id          : Block group ID string (blkgrpid)
    - county_fips    : County FIPS string (e.g. '075' for SF, '001' for Alameda)
    - min_area_ratio : Drop blocks smaller than this share of the block group (default 0.1%)
    """
    bg = gdf[gdf['blkgrpid'] == bg_id]
    if bg.empty:
        print(f"Block Group {bg_id} not found.")
        return

    bg_geom  = bg.geometry.iloc[0]
    bg_total = bg['off_all'].iloc[0]
    bg_area  = bg_geom.area

    # Clip county blocks to this block group
    county_blocks = all_blocks[all_blocks['COUNTYFP10'] == county_fips]
    blocks = county_blocks.clip(bg_geom).copy()

    # Areal interpolation
    blocks['area_ratio']     = blocks.geometry.area / bg_area
    blocks['area_ratio_pct'] = blocks['area_ratio'] * 100
    blocks['est_off_all']    = blocks['area_ratio'] * bg_total
    blocks = blocks[blocks['area_ratio'] > min_area_ratio]

    # Display table
    print(f"Block Group: {bg_id} | Total Off-Street Parking: {bg_total}")
    with pd.option_context('display.float_format', '{:.2f}'.format, 'display.max_rows', None):
        display(blocks[['GEOID10', 'area_ratio_pct', 'est_off_all']]
                .sort_values('est_off_all', ascending=False))

    # Map: parent block group (grey) + blocks (heatmap)
    m = bg.explore(
        color="grey",
        style_kwds={'fillOpacity': 0.1, 'weight': 2},
        name="Parent Block Group",
        tiles="CartoDB positron"
    )
    blocks.explore(
        m=m,
        column='est_off_all',
        cmap='YlGnBu',
        name="Estimated Parking per Block",
        tooltip=['GEOID10', 'est_off_all', 'area_ratio_pct'],
        popup=True,
        style_kwds={'fillOpacity': 0.7}
    )
    folium.LayerControl().add_to(m)
    display(m)

In [ ]:
# Case Study: Portsmouth Square Garage, San Francisco County
areal_interpolation(bg_id='060750611002', county_fips='075')

In [ ]:
# Case Study: UCSF Garage (Mission Bay), San Francisco County
areal_interpolation(bg_id='060750607001', county_fips='075')

In [ ]:
# Case Study: Parking Lots in West Berkeley, Alameda County
areal_interpolation(bg_id='060014220002', county_fips='001', min_area_ratio=0.0001)